# 05 — Reflection Loops: Confidence-Based Self-Improvement

## What is a Reflection Loop?

A **reflection loop** allows a node to automatically re-evaluate and improve its own output when the confidence score is below a defined threshold.

```
  Agent produces output
        │
        ▼
  confidence >= threshold?
        │
   YES  │  NO
        │   └──► Critic Agent reviews output
        │               │
        │        Agent revises with critic feedback
        │               │
        │        rounds_used < max_rounds?
        │               │
        │          YES  │  NO
        │               └──► emit final output (best attempt)
        ▼
  Emit output to next node
```

## Configuration

On any `GraphNodeBuilder`:
```python
.reflect(
    threshold=0.85,          # minimum confidence to skip reflection
    max_rounds=3,            # maximum reflection iterations
    critic="EchoAgent"       # critic agent name (default: CritiqueAgent)
)
```

## Notebook Goal

We set a **very high threshold (0.99)** on `SkillMatcherAgent` so that it is almost certain to trigger multiple reflection rounds. After the workflow completes, we examine `metrics.reflections_triggered` to confirm the loops ran.

In [ ]:
import time
import pprint

from multigen import SyncMultigenClient
from multigen.dsl import GraphBuilder

ORCHESTRATOR_URL = "http://localhost:8009"
client = SyncMultigenClient(base_url=ORCHESTRATOR_URL, timeout=120.0)
assert client.ping(), "Orchestrator not reachable."
print("Connected.")

## 1. Build the Graph with Reflection Enabled

The key call is `.reflect(threshold=0.99, max_rounds=3, critic="EchoAgent")`.

- `threshold=0.99` — only 99%+ confident outputs skip reflection (very hard to achieve)
- `max_rounds=3` — at most 3 reflection iterations before emitting the best output
- `critic="EchoAgent"` — the critic agent that reviews output and provides feedback

In [ ]:
graph_def = (
    GraphBuilder()
    # First node: standard parsing, no reflection needed
    .node("parse_resume")
        .agent("ResumeParserAgent")
        .params(output_format="json")
        .timeout(30)
        .done()
    # Second node: skill matching with aggressive reflection
    # The high threshold (0.99) almost guarantees reflection rounds will fire
    .node("match_skills")
        .agent("SkillMatcherAgent")
        .params(job_title="Principal ML Engineer", strictness="high")
        .reflect(
            threshold=0.99,      # nearly impossible to satisfy → forces reflection
            max_rounds=3,        # up to 3 revision attempts
            critic="EchoAgent",  # critic agent that reviews and guides revision
        )
        .timeout(60)             # extra time for reflection rounds
        .done()
    # Final node: aggregate the (possibly improved) skill scores
    .node("aggregate_scores")
        .agent("ScoreAggregatorAgent")
        .params(weights={"skills": 1.0})
        .timeout(20)
        .done()
    .edge("parse_resume",  "match_skills")
    .edge("match_skills",  "aggregate_scores")
    .entry("parse_resume")
    .max_cycles(15)
    .build()
)

# Show reflection config for match_skills node
match_node = next(n for n in graph_def["nodes"] if n["id"] == "match_skills")
print("match_skills node reflection config:")
print(f"  reflection_threshold : {match_node['reflection_threshold']}")
print(f"  max_reflections      : {match_node['max_reflections']}")
print(f"  critic_agent         : {match_node['critic_agent']}")

## 2. The Reflection Mechanism — Step by Step

During execution of `match_skills`:

1. `SkillMatcherAgent` runs and produces `{confidence: 0.73, matched_skills: [...]}`
2. Orchestrator sees `0.73 < 0.99` → reflection triggered
3. `EchoAgent` (critic) receives the output and produces a critique/feedback
4. `SkillMatcherAgent` reruns with the critique in context
5. Steps 2-4 repeat up to `max_rounds=3` times
6. After max rounds (or if confidence threshold is met), final output is emitted

**Each reflection counts as one `reflections_triggered` increment in metrics.**

## 3. Run the Workflow

In [ ]:
payload = {
    "resume_text": (
        "Carol Wang, 7 years ML, PyTorch, TensorFlow, CUDA, Kubernetes, "
        "MLflow. PhD Computer Science, Stanford 2017."
    ),
    "candidate_id": "candidate-004",
}

response = client.run_graph(graph_def=graph_def, payload=payload)
instance_id = response.instance_id
print(f"Launched — instance_id: {instance_id}")
print("Note: With threshold=0.99, this may take extra time for reflection rounds.")

## 4. Monitor Metrics During Execution

We poll `get_metrics()` while the workflow runs. Watch `reflections_triggered` increment as each reflection round fires.

In [ ]:
print(f"{'Time':>6s}  {'nodes_executed':>15s}  {'reflections_triggered':>22s}")
print("-" * 50)

start = time.time()
prev_reflections = -1

for _ in range(15):
    try:
        m = client.get_metrics(instance_id)
        elapsed = time.time() - start
        marker = " ← REFLECTION FIRED" if m.reflections_triggered > prev_reflections and prev_reflections >= 0 else ""
        print(f"{elapsed:6.1f}s  {m.nodes_executed:>15d}  {m.reflections_triggered:>22d}{marker}")
        prev_reflections = m.reflections_triggered

        # Stop polling once all 3 nodes have completed
        if m.nodes_executed >= 3:
            print("\nAll nodes executed — stopping poll.")
            break
    except Exception as e:
        print(f"  (metrics unavailable: {e})")
    time.sleep(3)

## 5. Inspect match_skills Node Output

The final output of `match_skills` should contain the best result after all reflection rounds. Look for:
- `confidence` — final confidence score after reflection
- `reflection_rounds` — how many rounds occurred (if the agent includes this)
- `revision_history` — the evolution of the output (if the agent includes this)

In [ ]:
time.sleep(3)  # ensure final state is persisted

try:
    ns = client.get_node_state(instance_id, "match_skills")
    print("match_skills final output:")
    pprint.pprint(ns.output, indent=2)
except Exception as e:
    print(f"Could not fetch node state: {e}")

## 6. Final Metrics — Confirm Reflections Triggered

In [ ]:
metrics = client.get_metrics(instance_id)

print("Final Metrics")
print("-" * 35)
print(f"  nodes_executed        : {metrics.nodes_executed}")
print(f"  nodes_skipped         : {metrics.nodes_skipped}")
print(f"  reflections_triggered : {metrics.reflections_triggered}")
print(f"  fan_outs_executed     : {metrics.fan_outs_executed}")
print(f"  circuit_breaker_trips : {metrics.circuit_breaker_trips}")
print(f"  error_count           : {metrics.error_count}")

if metrics.reflections_triggered > 0:
    print(f"\n✔ Reflection loops fired {metrics.reflections_triggered} time(s) as expected.")
else:
    print("\nNote: 0 reflections — the agent may have met the threshold or reflection is not yet implemented.")

## 7. Reflection Configuration Reference

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `threshold` | `float` | `0.0` | Minimum confidence to skip reflection (0.0 = never reflect) |
| `max_rounds` | `int` | `2` | Maximum reflection iterations per node execution |
| `critic` | `str` | `"CritiqueAgent"` | Name of the critic agent that reviews output |

**Practical thresholds:**
- `0.0` — no reflection (default)
- `0.7` — reflect on low-confidence outputs (most agents pass on first try)
- `0.9` — aggressive reflection (most outputs will be reflected once)
- `0.99` — near-guaranteed reflection (use in testing only)

**Next**: See `06_fan_out_consensus.ipynb` for parallel reasoning with consensus strategies.

In [ ]:
client.close()
print("Client closed.")